# Random Semantic Algebra — real fashion facets

Tests whether a **fixed random semantic coordinate system** can support cheap semantic boxes.

`productDisplayName → MiniLM embedding → random orthogonal rotation → 4-bit coordinates → sparse interval boxes`

Ground-truth facets are **never added to the embedding text**. We compare 4-bit boxes with 1-bit masks and a linear classifier, then test literal `A ∩ B` box intersections.

In [ ]:
!pip -q install -U datasets sentence-transformers scikit-learn pandas matplotlib pyarrow

In [ ]:
import numpy as np, pandas as pd, itertools, time, warnings
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, average_precision_score
from sklearn.linear_model import SGDClassifier
warnings.filterwarnings("ignore")
SEED=7; rng=np.random.default_rng(SEED); N=30000; K=28

# Real catalog; discard images before pandas conversion.
ds=load_dataset("ashraq/fashion-product-images-small",split="train")
cols=["id","gender","masterCategory","subCategory","articleType","baseColour","season","usage","productDisplayName"]
ds=ds.remove_columns([c for c in ds.column_names if c not in cols])
if len(ds)>N: ds=ds.select(rng.choice(len(ds),N,replace=False).tolist())
df=ds.to_pandas()
for c in cols:
    if c!="id": df[c]=df[c].fillna("Unknown").astype(str)
display(df.head())

# No facet leakage: semantic input is productDisplayName only.
m=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
X=m.encode(df.productDisplayName.tolist(),batch_size=256,show_progress_bar=True,
           normalize_embeddings=True,convert_to_numpy=True).astype("float32")
D=X.shape[1]

# Auto-pick common real concepts across fields.
fields=["baseColour","articleType","subCategory","usage","gender","masterCategory"]
cand=[]
for c in fields:
    for v,n in df[c].value_counts().items():
        p=n/len(df)
        if n>=350 and .025<=p<=.70: cand.append((c,v,n,p))
chosen=[]; used={c:0 for c in fields}
for z in sorted(cand,key=lambda x:x[2],reverse=True):
    if used[z[0]]<4: chosen.append(z); used[z[0]]+=1
    if len(chosen)==14: break
C=[(c,v) for c,v,_,_ in chosen]
display(pd.DataFrame(chosen,columns=["field","value","n","prevalence"]))
Y=np.column_stack([df[c].to_numpy()==v for c,v in C])

tr,te=train_test_split(np.arange(len(df)),test_size=.35,random_state=SEED)
Xr,Xe=X[tr],X[te]; Yr,Ye=Y[tr],Y[te]

# Fixed random orthogonal space: pre-quantization cosine is preserved exactly.
R,_=np.linalg.qr(rng.normal(size=(D,D)).astype("float32")); R=R.astype("float32")
Zr,Ze=Xr@R,Xe@R
Br,Be=Zr>=0,Ze>=0

# 4-bit per-coordinate quantization learned on train only.
lo=np.quantile(Zr,.005,axis=0); hi=np.quantile(Zr,.995,axis=0); span=np.maximum(hi-lo,1e-8)
def q4(Z): return np.rint((np.clip(Z,lo,hi)-lo)/span*15).astype("uint8")
Qr,Qe=q4(Zr),q4(Ze)
print("bytes/product: fp32",D*4,"1-bit",D/8,"4-bit",D/2)

# Geometry sanity check.
a=rng.integers(0,len(Xe),15000); b=rng.integers(0,len(Xe),15000)
cos=(Xe[a]*Xe[b]).sum(1)
ham=(Be[a]==Be[b]).mean(1)
l1=-np.abs(Qe[a].astype("int16")-Qe[b].astype("int16")).mean(1)
print("corr cosine↔1bit:",round(np.corrcoef(cos,ham)[0,1],4))
print("corr cosine↔4bit:",round(np.corrcoef(cos,l1)[0,1],4))

def best_th(s,y):
    best=(-1,None)
    for t in np.unique(np.quantile(s,np.linspace(.01,.99,100))):
        f=f1_score(y,s>=t,zero_division=0)
        if f>best[0]: best=(f,float(t))
    return best[1]

# Learn a sparse interval box: select random coords that separate + / -,
# retain central positive interval; all other coords are wildcards.
def learn_box(Q,y):
    P=Q[y].astype("float32"); N=Q[~y].astype("float32")
    sep=np.abs(P.mean(0)-N.mean(0))/np.sqrt(P.var(0)+N.var(0)+1e-4)
    idx=np.argsort(sep)[-K:]
    L=np.zeros(D,dtype="uint8"); U=np.full(D,15,dtype="uint8"); A=np.zeros(D,bool)
    L[idx]=np.floor(np.quantile(P[:,idx],.08,axis=0)).astype("uint8")
    U[idx]=np.ceil(np.quantile(P[:,idx],.92,axis=0)).astype("uint8"); A[idx]=1
    return L,U,A
def bscore(Q,B):
    L,U,A=B
    return ((Q[:,A]>=L[A])&(Q[:,A]<=U[A])).mean(1)
def learn_bits(B,y):
    d=B[y].mean(0)-B[~y].mean(0); ix=np.argsort(np.abs(d))[-K:]
    return ix,d[ix]>=0
def bitscore(B,M):
    ix,p=M; return (B[:,ix]==p).mean(1)

boxes=[learn_box(Qr,Yr[:,j]) for j in range(len(C))]
rows=[]
for j,(c,v) in enumerate(C):
    s4r=bscore(Qr,boxes[j]); s4e=bscore(Qe,boxes[j]); p4=s4e>=best_th(s4r,Yr[:,j])
    bm=learn_bits(Br,Yr[:,j]); s1r=bitscore(Br,bm); s1e=bitscore(Be,bm); p1=s1e>=best_th(s1r,Yr[:,j])
    clf=SGDClassifier(loss="log_loss",class_weight="balanced",max_iter=700,random_state=SEED).fit(Xr,Yr[:,j])
    lr=clf.predict_proba(Xr)[:,1]; le=clf.predict_proba(Xe)[:,1]; pl=le>=best_th(lr,Yr[:,j])
    rows.append([f"{c}={v}",Ye[:,j].sum(),
                 f1_score(Ye[:,j],p4),average_precision_score(Ye[:,j],s4e),
                 f1_score(Ye[:,j],p1),f1_score(Ye[:,j],pl)])
single=pd.DataFrame(rows,columns=["concept","test_n+","4bit_F1","4bit_AP","1bit_F1","linear_F1"])
display(single.round(3))
print("mean F1:",single[["4bit_F1","1bit_F1","linear_F1"]].mean().round(3).to_dict())


In [ ]:
# Literal box intersection: max lower bound, min upper bound.
def intersect(A,B):
    La,Ua,Aa=A; Lb,Ub,Ab=B
    act=Aa|Ab; L=np.maximum(La,Lb); U=np.minimum(Ua,Ub)
    return L,U,act,act&(L>U)

def iscore(Q,M):
    L,U,A,_=M
    return ((Q[:,A]>=L[A])&(Q[:,A]<=U[A])).mean(1)

pairs=[]
for a,b in itertools.combinations(range(len(C)),2):
    if C[a][0]==C[b][0]: continue
    yr=Yr[:,a]&Yr[:,b]; ye=Ye[:,a]&Ye[:,b]
    if yr.sum()<120 or ye.sum()<60: continue
    M=intersect(boxes[a],boxes[b])
    sr,se=iscore(Qr,M),iscore(Qe,M); pred=se>=best_th(sr,yr)
    # Very cheap fuzzy AND over already-compiled boxes.
    fr=np.minimum(bscore(Qr,boxes[a]),bscore(Qr,boxes[b]))
    fe=np.minimum(bscore(Qe,boxes[a]),bscore(Qe,boxes[b]))
    fp=fe>=best_th(fr,yr)
    pairs.append([
        f"{C[a][0]}={C[a][1]}",f"{C[b][0]}={C[b][1]}",int(ye.sum()),
        int((boxes[a][2]&boxes[b][2]).sum()),int(M[3].sum()),
        f1_score(ye,pred),average_precision_score(ye,se),
        f1_score(ye,fp),average_precision_score(ye,fe)
    ])

pairs=pd.DataFrame(pairs,columns=[
    "A","B","test_n+","shared_coords","empty_coord_conflicts",
    "literal_F1","literal_AP","soft_AND_F1","soft_AND_AP"
]).sort_values("test_n+",ascending=False).head(15)

display(pairs.round(3))
print("mean:",pairs[["literal_F1","soft_AND_F1","literal_AP","soft_AND_AP"]].mean().round(3).to_dict())

print(
"""READOUT

Good sign:
  4-bit boxes retain a useful fraction of linear F1 and AND queries compose.

Bad sign:
  linear F1 is high but boxes collapse => semantics survived rotation but are not axis-aligned.

If bad, next test:
  mixtures of 2-8 boxes/concept, weighted intervals, or a learned near-random rotation.
"""
)
